In [17]:
import numpy as np
import random

# Primary task

In [18]:
def arrivals_in_day(rate, t, Idx_for_process): 
    # Input: rate for arrival time, t is the day in the year, Idx_for_process is the type of patient
    # Output: list of a tuples with patient type in first entry and arrivaltime in the second entry. 


    # Initialize start of day and patients.
    time = 0
    patients = []

    # Let 0 patients arrive if rate is 0
    if rate <=0: 
        return []

    
    while True:
        time += np.random.exponential(1 / rate)

        # Check we are still within one day
        if time > 1:
            break

        # Append patient type and time for arrival
        patients.append((Idx_for_process, t + time))


    return patients

def arrivals_year(lam1, lam2,lam3):
    # Input: lami is the arrival rate function for ward i. 
    # Output: A list of tuples where the first entry in the tuple is the patient type and the last entry is the arrival time.
    
    # Initialize
    t =0
    Patients_1 = []
    Patients_2 = []
    Patients_3 = []

    #Iterate over the days
    while t < 365: 
        # Find rates
        rate1 = lam1(t)
        rate2 = lam2(t)
        rate3 = lam3(t)

        # Simulate arrivals for all three patient types. 
        Patients_1.extend(arrivals_in_day(rate1,t,1))
        Patients_2.extend(arrivals_in_day(rate2,t,2))
        Patients_3.extend(arrivals_in_day(rate3,t,3))

        t+=1

    # Merge list to create one list of all arrivals in a year
    All_patients = sorted(Patients_1 + Patients_2 + Patients_3, key=lambda x: x[1])
    return All_patients


In [19]:
def lam1(t): 
   return -(1/3650)*t**2 + (1/10)*t

def lam2(t): 
   return lam1(t)/5

def lam3(t):
   return 6


X = arrivals_year(lam1,lam2,lam3)

In [20]:
X[-1]

(3, 364.9375192782463)

In [21]:
# System of wards
def system(bedsA,bedsB,bedsC, patientflow_year):
    # Input: bedsA is number of beds in ward A,bedsB is number of beds in ward B, bedsC is number of beds in ward C. Patient_flow_year is a list of patients arriving in a year, where each entry in the list is a tuple containing the patient type and their arrival time. 
    # Output: blocked_i is the number of relocated patients for ward i and np.mean(bed_frac_i)/bedsi is the mean value of the fraction of beds in use in ward i

    #Initialize
    blocked_A =0
    blocked_B = 0
    blocked_C = 0

    beds_A = np.zeros(bedsA)
    beds_B = np.zeros(bedsB)
    beds_C = np.zeros(bedsC)

    bed_frac_A = []
    bed_frac_B = []
    bed_frac_C = []

    # Iterate through all patients
    for type, t in patientflow_year:
        # Release beds if time has passed of arrivaltime+LOS
        beds_A[beds_A <= t] = 0
        beds_B[beds_B <= t] = 0
        beds_C[beds_C <= t] = 0


        # Find idle beds
        idle_beds_A = np.where(beds_A == 0)[0]
        idle_beds_B = np.where(beds_B == 0)[0]
        idle_beds_C = np.where(beds_C == 0)[0]

        # Append number of beds in use
        bed_frac_A.append(bedsA-len(idle_beds_A))
        bed_frac_B.append(bedsB-len(idle_beds_B))
        bed_frac_C.append(bedsC-len(idle_beds_C))


        # Patients in ward A
        if type ==1: 
        # Find Length-of-Stay
            LOS = np.random.lognormal(np.log(4*np.sqrt(2)),np.log(2))
            # Check for idle beds
            if len(idle_beds_A) > 0:
                bed_id = idle_beds_A[0]
                beds_A[bed_id] = t + LOS

            else:
                # Reallocate patient
                blocked_A += 1
            
        # Patients in ward B
        elif type ==2: 
            # Find Length-of-Stay
            LOS = np.random.lognormal(np.log(6*np.sqrt(2)),np.log(2))
            # Check for idle beds
            if len(idle_beds_B) > 0:
                bed_id = idle_beds_B[0]
                beds_B[bed_id] = t + LOS

            else:
                # Increase blocked patients in B, and reallocate patient to A.
                blocked_B += 1
                if len(idle_beds_A)>0:
                    bed_id = idle_beds_A[0]
                    beds_A[bed_id] = t + LOS
                else: 
                    # If no space in A, randomly choose a patient in A to reallocate
                    blocked_A +=1
                    bed_id = np.random.choice(len(beds_A))
                    beds_A[bed_id] = t + LOS


        # Patients in ward C
        else: 
            # Find Length-of-Stay
            LOS = np.random.lognormal(np.log(5*np.sqrt(2)),np.log(2))
            # Check for idle beds
            if len(idle_beds_C) > 0:
                bed_id = idle_beds_C[0]
                beds_C[bed_id] = t + LOS

            else:
                # Reallocate patient
                blocked_C += 1
                
    return blocked_A,blocked_B,blocked_C, np.mean(bed_frac_A)/bedsA,np.mean(bed_frac_B)/bedsB,np.mean(bed_frac_C)/bedsC


# Sensitivity Analysis

## Control variates

In [22]:
# We make a monte carlo estimator just for sum of reallocated patients. 
# This is monte carlo
def sum_relocated(bedsA,bedsB,bedsC,patient_flows):
    # Input: bedsA is number of beds in ward A,bedsB is number of beds in ward B, bedsC is number of beds in ward C.patient_flows is a list of simulated yearly patient flow.
    # Output: mean and variance of the sum of reallocated patients across wards A, B and C
    A = []
    B = []
    C = []

    for X in patient_flows:
        a, b, c, bed_frac_A,bed_frac_B,bed_frac_C = system(bedsA,bedsB,bedsC, X)
        A.append(a)
        B.append(b)
        C.append(c)
    all = np.array(A)+np.array(B)+np.array(C)
    return np.mean(all), np.var(all)

In [23]:
# Modify existing system, in order to get output needed for control variate
def system_control(bedsA,bedsB,bedsC, patientflow_year):
    # Input: bedsA is number of beds in ward A,bedsB is number of beds in ward B, bedsC is number of beds in ward C. Patient_flow_year is a list of patients arriving in a year, where each entry in the list is a tuple containing the patient type and their arrival time. 
    # Output: blocked_i is the number of relocated patients for ward i and LOS_i is the mean value of LOS in use in ward i
    blocked_A =0
    blocked_B = 0
    blocked_C = 0

    beds_A = np.zeros(bedsA)
    beds_B = np.zeros(bedsB)
    beds_C = np.zeros(bedsC)

    LOS_A = []
    LOS_B = []
    LOS_C = []

    for type, t in patientflow_year:
        # Release beds if time has passed
        beds_A[beds_A <= t] = 0
        beds_B[beds_B <= t] = 0
        beds_C[beds_C <= t] = 0

        idle_beds_A = np.where(beds_A == 0)[0]
        idle_beds_B = np.where(beds_B == 0)[0]
        idle_beds_C = np.where(beds_C == 0)[0]

        if type ==1: 
            LOS = np.random.lognormal(np.log(4*np.sqrt(2)),np.log(2))
            if len(idle_beds_A) > 0:
                bed_id = idle_beds_A[0]
                beds_A[bed_id] = t + LOS
                LOS_A.append(LOS)

            else:
                blocked_A += 1
            

        elif type ==2: 
            LOS = np.random.lognormal(np.log(6*np.sqrt(2)),np.log(2))
    
            if len(idle_beds_B) > 0:
                bed_id = idle_beds_B[0]
                beds_B[bed_id] = t + LOS
                LOS_B.append(LOS)

            else:
                blocked_B += 1
                LOS_B.append(LOS)
                # SKal rykke B over i A og hvis A er fuld incremente at nogle er blevet afvist fra A.
                if len(idle_beds_A)>0:
                    bed_id = idle_beds_A[0]
                    beds_A[bed_id] = t + LOS
                    
                else: 
                    # Vælger randomly (er det rigtigt???), hvem der skal smides ud af A...
                    blocked_A+=1
                    bed_id = np.random.choice(len(beds_A))
                    beds_A[bed_id] = t + LOS
        
        else: 
            LOS = np.random.lognormal(np.log(5*np.sqrt(2)),np.log(2))
            if len(idle_beds_C) > 0:
                bed_id = idle_beds_C[0]
                beds_C[bed_id] = t + LOS
                LOS_C.append(LOS)


            else:
                blocked_C += 1
                
    return blocked_A,blocked_B,blocked_C, np.mean(LOS_A),np.mean(LOS_B), np.mean(LOS_C)

# Control variate function
def control_variate_sum(bedsA, bedsB, bedsC,patient_flows):
    # Input: bedsA is number of beds in ward A,bedsB is number of beds in ward B, bedsC is number of beds in ward C. patient_flows is a list of simulated yearly patient flow.
    # Output: mean and variance of the sum of reallocated patients across ward A, B and C. 
    #### Find ci
    # Initialize
    A = []
    B = []
    C = []

    LA = []
    LB = []
    LC = []

    n = int(len(patient_flows)/2)

    # Iterate
    for X in patient_flows[:n]:
        a,b,c,la,lb,lc = system_control(bedsA,bedsB,bedsC, X)
        A.append(a)
        B.append(b)
        C.append(c)
        LA.append(la)
        LB.append(lb)
        LC.append(lc)
    
    A = np.array(A)
    B = np.array(B)
    C = np.array(C)
    LA = np.array(LA)
    LB = np.array(LB)
    LC = np.array(LC)

    # Find ci
    ca = -np.cov(A,LA)[0,1]/np.var(LA)
    cB = -np.cov(B,LB)[0,1]/np.var(LB)
    cC = -np.cov(C,LC)[0,1]/np.var(LC)

    # Reinitialize to actually find the control variates
    A = []
    B = []
    C = []

    LA = []
    LB = []
    LC = []

    # Iterate
    for X in patient_flows[n:]:
        a,b,c,la,lb,lc = system_control(bedsA,bedsB,bedsC, X)
        A.append(a)
        B.append(b)
        C.append(c)
        LA.append(la)
        LB.append(lb)
        LC.append(lc)
    
    A = np.array(A)
    B = np.array(B)
    C = np.array(C)
    LA = np.array(LA)
    LB = np.array(LB)
    LC = np.array(LC)

    # Find new variable
    Ya = A+ca*(LA-8) #REMEMBER TO CHANGE MEANS IF THEY CHANGE!!!!
    meanA = np.mean(Ya)

    YB = B+cB*(LB-12) #REMEMBER TO CHANGE MEANS IF THEY CHANGE!!!!
    meanB = np.mean(YB)

    YC = C+cC*(LC-10) #REMEMBER TO CHANGE MEANS IF THEY CHANGE!!!!
    meanC = np.mean(YC)



    return meanA+meanB+meanC, np.var(Ya)+np.var(YB)+np.var(YC)+2*np.cov(Ya,YB)[0,1]+2*np.cov(YB,YC)[0,1]+2*np.cov(Ya,YC)[0,1]

In [24]:
np.random.seed(42)
patient_flow = [arrivals_year(lam1, lam2,lam3) for _ in range(100)]
sum_relocated(15,15,45,patient_flow)

(2235.71, 6812.305899999999)

In [25]:

control_variate_sum(15,15,45,patient_flow)

(2435.1534343346384, 4921.528196046962)

In [26]:
# Reduction
(6812- 4290)/6812*100

37.02290076335878

## Optimization of number of beds

In [29]:
patient_flow_opt = [arrivals_year(lam1,lam2,lam3) for _ in range(10)]

In [32]:
# Minimize the sum of relocated patients for different bed distribution

best_bed= []

for run in range(10):

    best_val = float("inf")
    best_alloc = None

    for _ in range(100):
        A = random.randint(1, 73)
        B = random.randint(1, 75 - A)
        C = 75 - A - B

        if C <=0: 
            continue

        val,var = control_variate_sum(A, B, C,patient_flow_opt)

        if val < best_val:
            best_val = val
            best_alloc = (A, B, C)
        
    print(run)
    best_bed.append((best_alloc, best_val))

0
1
2
3
4
5
6
7
8
9


In [33]:
best_bed

[((10, 31, 34), 2261.2796525392064),
 ((56, 14, 5), 1611.4753946097708),
 ((68, 3, 4), 2003.7491168943504),
 ((65, 5, 5), 1973.3189101979845),
 ((38, 11, 26), 1758.8743185731987),
 ((20, 8, 47), 2341.2798863931284),
 ((37, 14, 24), 1976.5755161078207),
 ((31, 22, 22), 1922.7203833655476),
 ((27, 8, 40), 2075.3564231971245),
 ((54, 3, 18), 1740.7568883436263)]

In [35]:
def optimize_beds_stepwise(patient_data, start_dist=[30, 10, 35]):
    current_dist = list(start_dist)
    best_dist = list(start_dist)
    
    # Get baseline metrics using your updated function
    best_score, var = control_variate_sum(current_dist[0],current_dist[1],current_dist[2],patient_data)
    print(f"Starting Baseline {current_dist}: {best_score} total issues")

    improved = True
    while improved:
        improved = False
        neighbors = []
        for i in range(3):
            for j in range(3):
                # Ensure we don't drop a ward's bed count below 0
                if i != j and current_dist[i] > 0: 
                    test_dist = list(current_dist)
                    test_dist[i] -= 1
                    test_dist[j] += 1
                    neighbors.append(test_dist)
        
        for neighbor in neighbors:
            print(f"Testing adjustment: {neighbor}...")
            score, var = control_variate_sum(current_dist[0],current_dist[1],current_dist[2],patient_data)
            
            if score < best_score:
                best_score = score
                best_dist = neighbor
                improved = True
        
        if improved:
            current_dist = list(best_dist)
            print(f"Found better distribution: {current_dist} with {best_score} issues")
            
    return best_dist, best_score

# MANGLER AT BLIVE ÆNDRET...


In [47]:
patient_flow_opt = [arrivals_year(lam1,lam2,lam3) for _ in range(20)]
best_vals = []
best_scores = []
for i in range(10):
    opt_dist, opt_score = optimize_beds_stepwise(patient_flow_opt, start_dist=best_bed[i][0])
    best_vals.append(opt_dist)
    best_scores.append(opt_score)
    print(f"\nFinal Optimal Distribution: {opt_dist} with {opt_score} total problems")
    

Starting Baseline [10, 31, 34]: 2879.1748377197114 total issues
Testing adjustment: [9, 32, 34]...
Testing adjustment: [9, 31, 35]...
Testing adjustment: [11, 30, 34]...
Testing adjustment: [10, 30, 35]...
Testing adjustment: [11, 31, 33]...
Testing adjustment: [10, 32, 33]...
Found better distribution: [9, 32, 34] with 2635.764392132404 issues
Testing adjustment: [8, 33, 34]...
Testing adjustment: [8, 32, 35]...
Testing adjustment: [10, 31, 34]...
Testing adjustment: [9, 31, 35]...
Testing adjustment: [10, 32, 33]...
Testing adjustment: [9, 33, 33]...

Final Optimal Distribution: [9, 32, 34] with 2635.764392132404 total problems
Starting Baseline [56, 14, 5]: 2612.7757611916486 total issues
Testing adjustment: [55, 15, 5]...
Testing adjustment: [55, 14, 6]...
Testing adjustment: [57, 13, 5]...
Testing adjustment: [56, 13, 6]...
Testing adjustment: [57, 14, 4]...
Testing adjustment: [56, 15, 4]...
Found better distribution: [55, 14, 6] with 2409.319905849125 issues
Testing adjustment: 

In [45]:
best_scores

[1299.6732739075037,
 2371.39112139874,
 2483.554839028446,
 2083.1518653117296,
 2181.963028211991,
 1904.1119702123856,
 2123.6794800334123,
 2260.3458467807523,
 2092.9670176807062,
 2208.5722461639825]

In [46]:
best_vals

[[10, 32, 33],
 [56, 15, 4],
 [68, 3, 4],
 [64, 5, 6],
 [38, 11, 26],
 [19, 9, 47],
 [37, 13, 25],
 [32, 22, 21],
 [27, 8, 40],
 [56, 2, 17]]

In [48]:
best_scores

[2635.764392132404,
 2115.2334492772725,
 2685.091715037991,
 2588.656995589273,
 2319.216844688578,
 2229.5313691632996,
 2143.5850473534183,
 2259.236241122322,
 2263.0784779755872,
 2382.276886750197]

In [49]:
best_vals

[[9, 32, 34],
 [53, 14, 8],
 [67, 5, 3],
 [65, 5, 5],
 [37, 11, 27],
 [20, 7, 48],
 [37, 12, 26],
 [30, 22, 23],
 [27, 8, 40],
 [55, 3, 17]]